# Week 6 — Part 02: Sampling and compression for tabular data

**Estimated time:** 75–120 minutes

## What success looks like (end of Part 02)

- You can build a deterministic compressed table representation (same output for same seed).
- You can write a bounded JSON artifact suitable for an LLM under `output/`.
- You can extend compression with at least one extra summary (numeric stats or top categories).

### Checkpoint

After running this notebook, you should have:

- a printed JSON payload for a `CompressedTable`
- an `output/compressed_input.json` file

## Learning Objectives

- Explain why compression is required for large tables
- Build a compressed representation (stats + sample rows)
- Produce deterministic JSON artifacts for LLM input
- Add TODO exercises for improving compression

**Lab tutorial**: [02_sampling_compression.md](./02_sampling_compression.md) - Detailed walkthrough and learning objectives

### What this part covers
This notebook covers **sampling and compression** — how to reduce a large dataset into a compact representation that fits within an LLM's context window.

**The core problem:** A CSV with 10,000 rows might be 2 MB of text — far too large to send to an LLM. You need to extract the *essential information* while staying under ~2,000 tokens.

**What to compress into:**
- Shape (rows × columns)
- Column names and types
- Missing value counts per column
- A small random sample of rows (5–10)
- Optional: numeric stats (min/mean/max), top categories

**Key rule:** Use a fixed `seed` so the same dataset always produces the same sample. This makes your pipeline reproducible.

## Overview

You usually cannot send a full dataset to an LLM. Instead you send a compressed representation:

- descriptive stats
- missingness summary
- a small sample of rows
- detected anomalies

---

## Underlying theory: you are fitting information into a fixed budget

The model has a fixed context window, so your input must satisfy a budget constraint:

$$
C \ge T_{\text{prompt}} + T_{\text{table}} + T_{\text{output}}
$$

If your table is large, $T_{\text{table}}$ dominates. Compression reduces $T_{\text{table}}$ by replacing raw rows with summaries.

Practical implication: good compression keeps *the facts that matter for the task* while dropping redundant detail.

### What this cell does
Defines `CompressedTable` (the typed output schema) and `compress_table()` — the main compression function — then serializes it to JSON and writes `output/compressed_input.json`.

**Walk through `compress_table()`:**
1. `df.sample(n=min(sample_n, len(df)), random_state=seed)` — random sample with a fixed seed for reproducibility (we use `seed=42`, the same default as the lesson and the capstone template)
2. `df.dtypes.to_dict()` — column type map (e.g., `{"job_title": "object", "salary_usd": "float64"}`)
3. `df.isna().sum().to_dict()` — missing value counts per column
4. `int(...)` casts — pandas returns numpy integers which are not JSON-serializable

**Why `sort_keys=True` in `json.dumps()`?** Produces deterministic JSON output. Run twice on the same data → byte-for-byte identical file. This matters for diffing artifacts across runs.

**What to check:** Open `output/compressed_input.json` and estimate the token count (~4 chars per token). It should be well under 2,000 tokens.

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple


try:
    import pandas as pd
except Exception as e:  # pragma: no cover
    pd = None
    _pd_import_error = e


@dataclass
class CompressedTable:
    shape: Tuple[int, int]
    columns: List[str]
    dtypes: Dict[str, str]
    missing: Dict[str, int]
    sample_rows: List[Dict[str, object]]
    sample_seed: int


def compress_table(df, *, sample_n: int = 6, seed: int = 42) -> CompressedTable:
    sample = df.sample(n=min(sample_n, len(df)), random_state=seed) if len(df) > 0 else df
    return CompressedTable(
        shape=(int(df.shape[0]), int(df.shape[1])),
        columns=list(df.columns),
        dtypes={c: str(t) for c, t in df.dtypes.to_dict().items()},
        missing={c: int(v) for c, v in df.isna().sum().to_dict().items()},
        sample_rows=sample.to_dict(orient="records"),
        sample_seed=seed,
    )


def to_json(ct: CompressedTable) -> str:
    payload = {
        "shape": list(ct.shape),
        "columns": ct.columns,
        "dtypes": ct.dtypes,
        "missing": ct.missing,
        "sample_rows": ct.sample_rows,
        "sample_seed": ct.sample_seed,
    }
    return json.dumps(payload, indent=2, sort_keys=True)


OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


if pd is None:
    print("pandas not available:", _pd_import_error)
else:
    # A small job-postings-flavored table (the Week 6 capstone theme).
    df = pd.DataFrame(
        {
            "job_title": ["Data Analyst", "AI Engineer", "Data Analyst", "ML Engineer", "BI Developer"],
            "location": ["Remote US", "New York NY", "Remote US", "Austin TX", None],
            "salary_usd": [85000, 120000, 90000, None, 95000],
        }
    )
    ct = compress_table(df, sample_n=4, seed=42)
    payload_str = to_json(ct)
    print(payload_str)
    (OUTPUT_DIR / "compressed_input.json").write_text(payload_str, encoding="utf-8")
    print("wrote:", OUTPUT_DIR / "compressed_input.json")

### Compressing long text columns (job descriptions)

The table above had short fields. Real job postings carry most of their signal in a long `job_description` column — often 1,000–4,000 characters each. Sending every full description to the LLM blows the token budget almost immediately.

Instead, compress the text column:
- detect the text column (`job_description`)
- keep a small **deterministic** sample of representative descriptions, each **truncated**
- add cheap structure a sample can't capture: top job titles, top locations
- never send the full column in bulk

The next cell runs this on the **real** classroom sample at `data/sample_job_postings.csv` (60 tech/data postings sourced from a public LinkedIn dataset — see `data/README.md`).

In [ ]:
from typing import Any, Dict


def compress_text_column(df, text_col: str, sample_n: int = 5, max_chars: int = 300, seed: int = 42) -> Dict[str, Any]:
    """Compress a long free-text column into a small, deterministic summary."""
    non_null = df[text_col].dropna()
    sample = non_null.sample(n=min(sample_n, len(non_null)), random_state=seed) if len(non_null) else non_null
    return {
        "text_column": text_col,
        "non_empty_count": int(df[text_col].notna().sum()),
        "sample_texts": sample.astype(str).str.slice(0, max_chars).tolist(),
        "note": "Sampled deterministically and truncated; full descriptions are never sent in bulk.",
    }


def compress_job_postings(df) -> Dict[str, Any]:
    """Job-posting-specific compression for the Week 6 capstone theme."""
    def top_counts(col: str) -> Dict[str, int]:
        if col not in df.columns:
            return {}
        return {str(k): int(v) for k, v in df[col].value_counts().head(10).to_dict().items()}

    compressed: Dict[str, Any] = {
        "shape": [int(df.shape[0]), int(df.shape[1])],
        "columns": list(df.columns),
        "top_job_titles": top_counts("job_title"),
        "top_locations": top_counts("location"),
    }
    if "job_description" in df.columns:
        compressed["description_summary"] = compress_text_column(df, "job_description")
    return compressed


CSV_PATH = Path("data/sample_job_postings.csv")

if pd is None:
    print("pandas not available; skipping the real-data demo")
elif not CSV_PATH.exists():
    print("sample CSV not found:", CSV_PATH, "- run scripts/build_job_postings_sample.py first")
else:
    jobs = pd.read_csv(CSV_PATH)
    job_compressed = compress_job_postings(jobs)
    job_json = json.dumps(job_compressed, indent=2, sort_keys=True)
    (OUTPUT_DIR / "compressed_job_postings.json").write_text(job_json, encoding="utf-8")
    print(f"{len(jobs)} postings -> compressed JSON is {len(job_json)} chars")
    print(job_json[:700], "...")

### Bonus: keyword frequency hints (cheap skill evidence)

Sampling shows the model *what postings look like*; it does not tell it *which
skills recur*. A cheap, deterministic complement is a **keyword frequency hint**:
count how many postings mention each known skill. This is exactly the kind of
compact evidence the capstone's LLM step reasons over.

The full flow — evidence → prompt → validate → report — is covered in
[03_skill_analysis.md](./03_skill_analysis.md).

In [ ]:
import re

SKILL_KEYWORDS = ["python", "sql", "excel", "tableau", "power bi", "aws",
                  "azure", "spark", "docker", "machine learning", "communication"]


def skill_frequency_hints(df, text_col="job_description", keywords=SKILL_KEYWORDS):
    # Count how many postings mention each known skill (whole-word match).
    texts = df[text_col].dropna().astype(str).str.lower()
    counts = {kw: int(texts.str.contains(r"\b" + re.escape(kw) + r"\b", regex=True).sum())
              for kw in keywords}
    return {k: v for k, v in sorted(counts.items(), key=lambda kv: kv[1], reverse=True) if v}


if pd is None:
    print("pandas not available; skipping keyword hints")
elif "jobs" not in globals():
    print("run the job-postings compression cell above first")
else:
    # `jobs` was loaded in the job-postings compression cell above.
    print("skill keyword hints:", skill_frequency_hints(jobs))

### Estimating the token budget

Before sending anything to an LLM, estimate how many tokens the compressed object will cost. A rough rule is **~4 characters per token**. Keep the compressed data well under **~2,000 tokens** so the system prompt, task instructions, and the model's output budget all still fit inside the context window.

The next cell defines `estimate_tokens()` and checks both the small table payload and the real job-postings payload against that budget.

In [ ]:
def estimate_tokens(obj, chars_per_token: float = 4.0) -> int:
    """Rough token estimate for any JSON-serializable object (~4 chars per token)."""
    return int(len(json.dumps(obj)) / chars_per_token)


BUDGET = 2000  # keep compressed data well under this so prompt + output also fit

if pd is None:
    print("pandas not available; skipping token budget check")
else:
    checks = {"compressed_input (table)": json.loads(to_json(ct))}
    if "job_compressed" in globals():
        checks["compressed_job_postings"] = job_compressed

    for name, obj in checks.items():
        est = estimate_tokens(obj)
        flag = "OK" if est < BUDGET else "TOO BIG (reduce sample_n / truncate more)"
        print(f"{name}: ~{est} tokens [{flag}]")

### What this cell does
Defines two extension functions you need to implement:

- **`add_numeric_summary(df)`** — compute min/mean/max per numeric column. This helps the LLM understand value ranges without seeing every row.
- **`add_top_categories(df, top_k)`** — compute the most frequent values per categorical column. This reveals data distributions (e.g., "80% of rows have city='NY'").

**Why add these summaries?** A random sample of 5 rows might not show the full picture. Numeric stats and top categories give the LLM structural information that no sample can capture efficiently.

**Your task:** Implement both functions. Hints:
- `df.select_dtypes(include='number')` selects numeric columns
- `df[col].value_counts(dropna=False).head(top_k)` gives top categories including NaN

## Why the design choices matter

- sampling uses a `seed` so results are stable across runs
- `sort_keys=True` produces deterministic JSON (diff-friendly)
- a structured object (`CompressedTable`) makes it easier to evolve the contract later

Calibration tip:

- start with a small `sample_n` (e.g., 5–10)
- if the LLM misses important patterns, add targeted summaries rather than dumping more rows

In [ ]:
from typing import Any, Dict


def add_numeric_summary(df) -> Dict[str, Any]:
    # TODO: add min/mean/max per numeric column.
    return {}


def add_top_categories(df, top_k: int = 3) -> Dict[str, Any]:
    # TODO: compute top categories for object/string columns.
    return {}


if pd is None:
    print("pandas not available; skipping exercise cells")
else:
    extra = {
        "numeric_summary": add_numeric_summary(df),
        "top_categories": add_top_categories(df, top_k=3),
    }
    (OUTPUT_DIR / "compressed_extras.json").write_text(json.dumps(extra, indent=2, sort_keys=True), encoding="utf-8")
    print("wrote:", OUTPUT_DIR / "compressed_extras.json")

## Appendix: Solutions (peek only after trying)

Reference implementations for the TODO functions in this notebook.

In [ ]:
def add_numeric_summary(df) -> Dict[str, Any]:
    numeric = df.select_dtypes(include="number")
    out: Dict[str, Any] = {}
    if numeric.shape[1] == 0:
        return out
    for col in numeric.columns:
        s = numeric[col].dropna()
        if len(s) == 0:
            out[str(col)] = {"min": None, "mean": None, "max": None}
        else:
            out[str(col)] = {
                "min": float(s.min()),
                "mean": float(s.mean()),
                "max": float(s.max()),
            }
    return out


def add_top_categories(df, top_k: int = 3) -> Dict[str, Any]:
    obj = df.select_dtypes(include=["object"]) if hasattr(df, "select_dtypes") else df
    out: Dict[str, Any] = {}
    if getattr(obj, "shape", (0, 0))[1] == 0:
        return out

    for col in obj.columns:
        vc = obj[col].fillna("<NA>").astype(str).value_counts(dropna=False)
        top = vc.head(int(top_k))
        out[str(col)] = [{"value": str(idx), "count": int(cnt)} for idx, cnt in top.items()]
    return out


if pd is not None:
    extra_solution = {
        "numeric_summary": add_numeric_summary(df),
        "top_categories": add_top_categories(df, top_k=3),
    }
    (OUTPUT_DIR / "compressed_extras_solution.json").write_text(
        json.dumps(extra_solution, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    print("wrote:", OUTPUT_DIR / "compressed_extras_solution.json")